In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import glob
import pandas as pd
import json
import commentjson
import commentjson
import os
import re
from urllib.parse import urlencode
import requests
import datetime
import logging
from pathlib import Path
import time
from datetime import datetime, timedelta

import yaml
from pathlib import Path
import itertools
import numpy as np
from typing import Dict, List, Any

from IPython.display import display, HTML
import ipywidgets as ipw
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [5]:
from bloomberg.enterprise.oauth import (OAuthClient, EnvTier,
                                        OAuthDeviceModeConfig, OAuthServerModeConfig
                                        )

from bloomberg.enterprise.oauth.oauth_flows.oauth_user_mode import UserModeOauth
from bloomberg.enterprise.oauth.utils import _oauth_store

### Import Source Code

In [6]:
from src.api_config import (
    OPTIMIZATION_TRIGGER_ENDPOINT, RESULTS_RETRIEVAL_ENDPOINT,
    WORKFLOWS_PATH, WORKFLOW_RUNS_PATH,
    CATALOG_PATH, REPORT_PATH,
    CONNECTION_TEST_PATH, WAIT_TIME_SECONDS, 
    AUTH_CONFIG_PATH, get_authorization_headers, 
    test_connection, auth_manager
)
from src.config_loader import (
    load_optimization_config,
    generate_constraint_combinations,
    generate_risk_model_combinations,
    generate_all_parameter_combinations
)
from src.template_utils import (
    load_template, 
    populate_template, 
    load_constraint_mappings,
    load_goal_mappings,
    load_trade_universe_mappings,
    load_custom_instrument_list,
    map_constraint_to_params
)
from src.task_builder import build_task
from src.optimization_workflow import (
    register_optimization_tasks, build_optimization_request,
    get_optimization_response, run_pending_optimizations,
    generate_optimization_report, view_optimization_report
)
from src.optimization_tracker import OptimizationTracker
from src.analysis_utils import OptimizationAnalyzer
from src.workflow_manager import OptimizationWorkflowManager

from src.data_manager import OptimizationDataManager
from src.plot_manager import (ParallelCoordinatesPlotter, create_distribution_plot, build_security_trade_consistency_fig)
from src.gui import PortfolioOptimizationGUI

In [7]:
from src import plot_manager

### Test API Connection

In [9]:
# Test API Connection
if test_connection():
    print('Connection is fine')
    # Continue with your application
else:
    print('Connection failed - please check credentials and network')
    # Handle the connection failure

Connection is fine


In [8]:
pd.set_option('display.max_columns', None)

In [20]:
config = load_optimization_config("config/optimization_parameters.yaml")
all_combinations = generate_all_parameter_combinations(config)
constraint_combinations = generate_constraint_combinations(config)

In [12]:
len(all_combinations)

960

In [21]:
len(constraint_combinations)

480

In [12]:
config

{'portfolios': ['EQUITY8_US', 'EQUITY8_MID_CAP_GROWTH'],
 'benchmarks': ['RAY', 'RDG', 'RLV'],
 'trade_universes': ['benchmark_buy_and_sell', 'active_ideas_buy_and_sell'],
 'portfolio_benchmark_pairs': [{'portfolio': 'EQUITY8_US', 'benchmark': 'RAY'},
  {'portfolio': 'EQUITY8_MID_CAP_GROWTH', 'benchmark': 'RDG'}],
 'portfolio_benchmark_universe_mappings': [{'portfolio': 'EQUITY8_US',
   'benchmark': 'RAY',
   'universes': ['benchmark_buy_and_sell', 'active_ideas_buy_and_sell']},
  {'portfolio': 'EQUITY8_MID_CAP_GROWTH',
   'benchmark': 'RDG',
   'universes': ['benchmark_buy_and_sell', 'active_ideas_buy_and_sell']}],
 'risk_options': [{'model': ['US_EQUITY']},
  {'scaling': ['YEAR']},
  {'horizon': ['ANNUAL']}],
 'goals': ['minimize_active_total_risk', 'maximize_custom_expected_return'],
 'constraints': {'turnover': {'min': 0.2, 'max': 0.5, 'step': 0.1},
  'maximum_positions': {'values': [200, 250, 300]},
  'active_total_risk': {'min': 0.02, 'max': 0.05, 'step': 0.01},
  'benchmark_sect

### Initialize Tracker and Workflow Manager

In [13]:
tracker = OptimizationTracker()
workflow_mngr = OptimizationWorkflowManager()
analyzer = OptimizationAnalyzer(workflow_mngr.tracker)

#### Analysis

In [34]:
def create_optimization_summary_section(config_path="config/optimization_parameters.yaml"):
    # Load the configuration
    try:
        config = load_optimization_config(config_path)
    except Exception as e:
        return ipw.HTML(f"<div style='color: red'>Error loading configuration: {str(e)}</div>")
    
    # Create HTML content for the summary
    summary_html = """
    <div style="padding: 10px; font-family: Arial, sans-serif;">
        <h2 style="margin-top: 5px;">Optimization Task Configuration Summary</h2>
    """
    
    # Portfolio-Benchmark-Universe section
    summary_html += """
        <div style="margin-bottom: 20px;">
            <h3 style="margin-bottom: 10px; border-bottom: 1px solid #ddd; padding-bottom: 5px;">
                Portfolio-Benchmark-Universe Combinations
            </h3>
            <ul style="list-style-type: none; padding-left: 10px;">
    """
    
    # Add portfolio-benchmark mappings
    if 'portfolio_benchmark_universe_mappings' in config:
        for mapping in config['portfolio_benchmark_universe_mappings']:
            portfolio = mapping.get('portfolio', 'Unknown')
            benchmark = mapping.get('benchmark', 'Unknown')
            universes = ', '.join(mapping.get('universes', ['None']))
            summary_html += f"""
                <li style="margin-bottom: 8px;">
                    <b>Portfolio:</b> {portfolio} | 
                    <b>Benchmark:</b> {benchmark} | 
                    <b>Trade Universes:</b> {universes}
                </li>
            """
    summary_html += """
            </ul>
        </div>
    """
    
    # Risk Options section
    summary_html += """
        <div style="margin-bottom: 20px;">
            <h3 style="margin-bottom: 10px; border-bottom: 1px solid #ddd; padding-bottom: 5px;">
                Risk Model Options
            </h3>
            <ul style="list-style-type: none; padding-left: 10px;">
    """
    
    if 'risk_options' in config:
        for option in config['risk_options']:
            for key, values in option.items():
                values_str = ', '.join([str(v) for v in values]) if isinstance(values, list) else str(values)
                summary_html += f"""
                    <li style="margin-bottom: 8px;">
                        <b>{key.capitalize()}:</b> {values_str}
                    </li>
                """
    summary_html += """
            </ul>
        </div>
    """
    
    # Goals section
    summary_html += """
        <div style="margin-bottom: 20px;">
            <h3 style="margin-bottom: 10px; border-bottom: 1px solid #ddd; padding-bottom: 5px;">
                Optimization Goals
            </h3>
            <ul style="list-style-type: none; padding-left: 10px;">
    """
    
    if 'goals' in config:
        for goal in config['goals']:
            summary_html += f"""
                <li style="margin-bottom: 8px;">
                    <b>{goal}</b>
                </li>
            """
    summary_html += """
            </ul>
        </div>
    """
    
    # Constraints section - with improved styling and fixed widths
    summary_html += """
        <div style="margin-bottom: 20px;">
            <h3 style="margin-bottom: 10px; border-bottom: 1px solid #ddd; padding-bottom: 5px;">
                Constraints
            </h3>
            <table style="width: auto; border-collapse: collapse; table-layout: fixed; max-width: 800px;">
                <tr style="background-color: #d4e6f1;">
                    <th style="padding: 8px; text-align: left; border: 1px solid #ddd; width: 210px; font-weight: bold;">Constraint</th>
                    <th style="padding: 8px; text-align: left; border: 1px solid #ddd; width: 250px; font-weight: bold;">Range/Values</th>
                    <th style="padding: 8px; text-align: left; border: 1px solid #ddd; width: 100px; font-weight: bold;">Type</th>
                </tr>
    """

    if 'constraints' in config:
        for constraint_name, constraint_config in config['constraints'].items():
            constraint_type = "Discrete" if 'values' in constraint_config else "Range"
            
            if constraint_type == "Discrete":
                values_str = ', '.join([str(v) for v in constraint_config['values']])
            else:
                values_str = f"Min: {constraint_config.get('min', 'N/A')}, " + \
                            f"Max: {constraint_config.get('max', 'N/A')}, " + \
                            f"Step: {constraint_config.get('step', 'N/A')}"
            
            summary_html += f"""
                <tr>
                    <td style="padding: 8px; text-align: left; border: 1px solid #ddd;">{constraint_name}</td>
                    <td style="padding: 8px; text-align: left; border: 1px solid #ddd;">{values_str}</td>
                    <td style="padding: 8px; text-align: left; border: 1px solid #ddd;">{constraint_type}</td>
                </tr>
            """

    summary_html += """
            </table>
        </div>
    """
    
    # Additional information
    summary_html += """
        <div style="margin-bottom: 20px;">
            <h3 style="margin-bottom: 10px; border-bottom: 1px solid #ddd; padding-bottom: 5px;">
                Additional Settings
            </h3>
            <ul style="list-style-type: none; padding-left: 10px;">
    """
    
    additional_settings = {
        'As Of Dates': config.get('as_of_dates', ['None']),
        'Save To': config.get('saveTo', 'None'),
        'Enable Look Through': config.get('enableLookThrough', 'None'),
        'Infused Cash Amount': config.get('infusedCashAmount', 'None')
    }
    
    for setting, value in additional_settings.items():
        value_str = ', '.join([str(v) for v in value]) if isinstance(value, list) else str(value)
        summary_html += f"""
            <li style="margin-bottom: 8px;">
                <b>{setting}:</b> {value_str}
            </li>
        """
    
    summary_html += """
            </ul>
        </div>
    """
    
    # Close the main div
    summary_html += """
    </div>
    """
    
    # Calculate the number of portfolio-benchmark-universe combinations
    portfolio_benchmark_combinations = len(config.get('portfolio_benchmark_universe_mappings', []))

    # Calculate constraint combinations
    constraint_combinations = len(generate_constraint_combinations(config))
    
    # Risk model combinations
    risk_combinations = len(generate_risk_model_combinations(config))

    try:
        # Calculate total combinations using the actual method from config_loader
        all_combinations = generate_all_parameter_combinations(config)
        total_combinations = len(all_combinations)
    except Exception as e:
        # Total combinations fallback
        total_combinations = portfolio_benchmark_combinations * constraint_combinations * risk_combinations * len(config.get('as_of_dates', [1]))
    
    # Create a summary widget
    summary_widget = ipw.HTML(summary_html)
    
    # Create a statistics widget
    stats_html = f"""
    <div style="background-color: #f8f9fa; padding: 15px; border-radius: 5px; margin-top: 10px;">
        <h3 style="margin-top: 0;">Optimization Statistics</h3>
        <ul>
            <li><b>Portfolio-Benchmark-Universe Combinations:</b> {portfolio_benchmark_combinations}</li>
            <li><b>Constraint Permutations:</b> {constraint_combinations}</li>
            <li><b>Risk Model Variations:</b> {risk_combinations}</li>
            <li><b>As-of Dates:</b> {len(config.get('as_of_dates', [1]))}</li>
            <li><b>Total Optimization Tasks:</b> {total_combinations}</li>
        </ul>
    </div>
    """
    stats_widget = ipw.HTML(stats_html)
    
    # Create accordion for summary
    accordion = ipw.Accordion(children=[ipw.VBox([summary_widget, stats_widget])])
    accordion.set_title(0, '⚙️ Optimization Configuration Summary')
    
    # Start collapsed
    accordion.selected_index = None
    
    return accordion

In [35]:
create_optimization_summary_section()

Accordion(children=(VBox(children=(HTML(value='\n    <div style="padding: 10px; font-family: Arial, sans-serif…

<h2>1. Executive Summary Section</h2>

Optimization Health Panel: Key metrics showing completed vs. failed runs<br>
Performance Highlights: Top-performing optimization configurations<br>
Key Metrics Comparison: Active return vs. active risk scatter plot with efficient frontier overlay

<h3>A. Status Overview Donut Chart</h3>

Central donut chart showing the proportion of:

Successful runs (green)<br>
Failed runs (red)<br>
Running/pending runs (blue)<br>
Not yet started (gray)


Large percentage in the center showing success rate


In [14]:
optimization_status_frame = tracker.index_df.groupby('status').count()['optimization_id'].to_frame()
plot_manager.build_optimization_health_fig(optimization_status_frame)

FigureWidget({
    'data': [{'hole': 0.7,
              'labels': [FAILED, success],
              'marker': {'colors': ['#e74c3c', '#2ecc71']},
              'pull': [0.02, 0],
              'textinfo': 'label+percent',
              'textposition': 'outside',
              'type': 'pie',
              'uid': '4683b2cc-0bb1-46ea-bf7e-76d3062efef7',
              'values': [7, 1209]}],
    'layout': {'annotations': [{'font': {'color': '#2c3e50', 'size': 40},
                                'showarrow': False,
                                'text': '99.4%',
                                'x': 0.5,
                                'y': 0.5},
                               {'font': {'color': '#7f8c8d', 'size': 16},
                                'showarrow': False,
                                'text': 'Success Rate',
                                'x': 0.5,
                                'y': 0.4}],
               'height': 500,
               'legend': {'orientation': 'h', 'x': 0.

<h3>B. Summary Statistics Bar</h3>

Total optimizations registered<br>
Total completed<br>
Total pending<br>
Average completion time

Same as above by constraint threhsold - grid

In [12]:
# Total optimizations registered<br>
# Total completed<br>
# Total pending<br>

In [15]:
num_registered_optimizations = tracker.index_df.shape[0]
num_successful_optimizations = tracker.get_successful_optimizations().shape[0]
num_pending_optimizations = len(tracker.get_pending_optimizations())
num_failed_optimizations = len(tracker.get_failed_optimizations())

In [16]:
num_registered_optimizations_html = ipw.HTML(f"<h3># of Registered Optimizations: {num_registered_optimizations}</h3>", layout={'height': '35px'})
num_successful_optimizations_html = ipw.HTML(f"<h3># of Successful Optimizations: {num_successful_optimizations}</h3>", layout={'height': '35px'})
num_pending_optimizations_html = ipw.HTML(f"<h3># of Pending Optimizations: {num_pending_optimizations}</h3>", layout={'height': '35px'})
num_failed_optimizations_html = ipw.HTML(f"<h3># of Failed Optimizations: {num_failed_optimizations}</h3>", layout={'height': '35px'})

In [17]:
ipw.HBox([
    num_registered_optimizations_html,
    num_successful_optimizations_html,
    num_pending_optimizations_html,
    num_failed_optimizations_html,
], layout={'justify_content':'space-between', 'padding':'0px 100px 0px 100px'})

In [18]:
constraints = ['max_active_risk','max_positions','max_turnover', 'max_security_weight']
constraint_combinations_frame = pd.pivot_table(
    data=tracker.index_df,
    index='status',
    columns=constraints,
    values='optimization_id',
    aggfunc='count',
    fill_value=0
).T

# Have accordions for active risk enums
# Grid within each of max positions vs max turnover

active_risk_enums = sorted(constraint_combinations_frame.reset_index(level=0)['max_active_risk'].unique())
combination_frames = []
for active_risk in active_risk_enums:
    combination_frames.append(ipw.HTML(constraint_combinations_frame.loc[(active_risk),:].to_html()))

constraint_combo_accordions = ipw.Accordion(children=combination_frames)
for i, df in enumerate(combination_frames):
    constraint_combo_accordions.set_title(i, f"Active Total Risk = {active_risk_enums[i]}")

constraint_combo_accordions.selected_index = None

constraint_combo_summary_html = ipw.HTML("<h2>Optimization Runs by Constraint Thresholds</h2>")


ipw.VBox([constraint_combo_summary_html, constraint_combo_accordions])

<h3>C. Recent Activity Timeline</h3>

Mini-timeline showing the last 5-10 optimization runs<br>
Color-coded status markers<br>
Hover tooltips with optimization IDs and error messages (if failed)

In [19]:
recent_activity_frame = tracker.index_df.copy()
plot_manager.build_recent_optimization_activity_plot(recent_activity_frame)

FigureWidget({
    'data': [{'hoverinfo': 'text',
              'hovertext': ('ID: opt_EQUITY8_US_RAY_benchma' ... "s<br>Error: 'errorDescription'"),
              'marker': {'color': '#2ecc71', 'line': {'color': 'white', 'width': 2}, 'size': 19, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'success',
              'showlegend': False,
              'type': 'scatter',
              'uid': 'ae0b78ee-b87e-46bb-8187-180e48d049c3',
              'x': [2025-04-22 16:34:22.198155],
              'y': [1]},
             {'hoverinfo': 'text',
              'hovertext': ('ID: opt_EQUITY8_MID_CAP_GROWTH' ... '22 15:32:01<br>Status: success'),
              'marker': {'color': '#2ecc71', 'line': {'color': 'white', 'width': 2}, 'size': 19, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'success',
              'showlegend': False,
              'type': 'scatter',
              'uid': '80fcebd7-58d5-45fa-b70c-b6e32a645cf8',
              '

<h3>D. Alert Banner</h3>

Red/yellow/green status bar<br>
Displays critical issues or warnings<br>
Examples: "3 optimizations failed in the last hour", "API connection unstable"


In [20]:
failed_optimizations_frame = pd.DataFrame(tracker.get_failed_optimizations())
alert_container = ipw.Accordion(children=[ipw.HTML(failed_optimizations_frame.to_html())])
alert_container.set_title(0,'Optimization Run Errors')
alert_container.selected_index = None
alert_container

Accordion(children=(HTML(value='<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: ri…

<h3>E. Quick Actions</h3>

"Run Pending" button (with count of pending jobs)<br>
"Rerun Failed" button (with count of failed jobs)<br>
"Export Log" button

In [21]:
run_pending_btn = ipw.Button(description='Run Pending Optimizations', layout={'width':'initial'}, button_style='info')
run_failed_btn = ipw.Button(description='Run Failed Optimizations', layout={'width':'initial'}, button_style='warning')

ipw.HBox([run_pending_btn, run_failed_btn])

<h3>F. Performance Trend Sparkline</h3>

Small inline chart showing success rate trend over the last 24 hours or 7 days<br>
Simple line with colored background gradient

In [22]:
frame = tracker.index_df.copy()
plot_manager.build_cumulative_optimizations_plot(frame)

FigureWidget({
    'data': [{'type': 'scatter',
              'uid': 'b480af8b-395f-4c91-aa33-a5083b0f6bd0',
              'x': array(['2025-04-19T19:18:02.481667', '2025-04-19T19:20:27.156098',
                          '2025-04-19T19:22:29.899788', ..., '2025-04-22T15:31:36.633944',
                          '2025-04-22T15:32:01.928701', '2025-04-22T16:34:22.198155'],
                         shape=(1216,), dtype=object),
              'y': array([   0,    1,    2, ..., 1213, 1214, 1215], shape=(1216,))}],
    'layout': {'height': 450,
               'margin': {'b': 50, 'l': 100, 'r': 50, 't': 80},
               'template': '...',
               'title': {'text': 'Cumulative Optimization Runs',
                         'x': 0.5,
                         'xanchor': 'center',
                         'y': 0.95,
                         'yanchor': 'top'},
               'width': 700}
})

<h3>G. Build the Summary Dashboard</h3>

In [23]:
ipw.VBox([
    ipw.HBox([
        # Optimization Health
        plot_manager.build_optimization_health_fig(optimization_status_frame),
        ipw.HBox(layout={'width':'40px'}),
        # Optimization Runs by Constraint Thresholds
        ipw.VBox(
            [
                constraint_combo_summary_html, 
                ipw.VBox(layout={'height':'15px'}), 
                constraint_combo_accordions]
        ), 
        ipw.HBox(layout={'width':'40px'}),
        # Cumulative Optimization Runs
        plot_manager.build_cumulative_optimizations_plot(frame),
    ], layout={'align_items':'flex-start', 'justify_content':'space-between', 'padding':'0px 100px 0px 0px'}),
    ipw.VBox(layout={'height':'20px'}),
    # Optimization Tracker Stats
    ipw.HBox([
            num_registered_optimizations_html,
            num_successful_optimizations_html,
            num_pending_optimizations_html,
            num_failed_optimizations_html,
        ], layout={'justify_content':'space-between', 'padding':'0px 100px 0px 100px'}),
    ipw.VBox(layout={'height':'40px'}),
    # Recent Activity
    ipw.VBox([plot_manager.build_recent_optimization_activity_plot(recent_activity_frame)], layout={'border': '1px solid lightgrey', 'padding':'0px 100px 0px 20px'}),
    ipw.VBox(layout={'height':'20px'}),
    # Optimization Run Errors
    ipw.VBox([alert_container], layout={'padding':'0px 50px 0px 50px'}),
    ipw.VBox(layout={'height':'20px'}),
    # Optmization Run Buttons
    ipw.HBox([run_pending_btn, run_failed_btn], layout={'padding':'0px 50px 0px 50px'}),
    ipw.VBox(layout={'height':'20px'}),
])

    'data': [{'hole': 0.7,
              'labels': [FAILED, succe…

In [28]:
from src.optimization_exec_summary import OptimizationExecutiveDashboard

In [29]:
opt_summary_dash = OptimizationExecutiveDashboard(tracker)

In [30]:
opt_summary_dash.get_view()

    'data': [{'hole': 0.7,
              'labels': [FAILED, succe…

<h2>2. Constraint Sensitivity Analysis</h2>

Constraint Impact Heatmap: Visual representation of how each constraint affects key metrics<br>
Binding Constraints Tracker: Identifies which constraints are consistently binding

In [22]:
custom_goal_mapper = {
    "CUSTOM_NUMBER(FIELD = 'CF_7459422178055815169')": "expected_return",
    "active_total_risk": "active_total_risk"
}

constraint_description_mapper = {
    'max_active_risk': 'Max Active Total Risk',
    'max_positions': 'Max Positions',
    'max_turnover': 'Max Turnover',
    'max_sector_deviation': 'Max Sector Deviation',
    'max_security_weight': 'Max Security Weight'
}

goal = "CUSTOM_NUMBER(FIELD = 'CF_7459422178055815169')"
#goal = "active_total_risk"
portfolio = 'EQUITY8_US'

In [23]:
# sensitivity_frames = analyzer.load_goal_constraint_sensitivity_frames(goal, list(constraint_description_mapper.keys()), portfolio)

In [24]:
from src.constraint_sensitivity_analysis_view import ConstraintSensitivityView

In [25]:
constraint_sensitivity_analysis_view = ConstraintSensitivityView(analyzer, portfolio, goal, custom_goal_mapper, constraint_description_mapper)

In [26]:
constraint_sensitivity_analysis_view.get_view()

<h2>3. Portfolio Characteristic Analysis</h2>

Stock Contribution Analysis: Identify consistently selected/excluded securities

In [31]:
successful_optimizations_frame = tracker.get_successful_optimizations().copy()

In [32]:
successful_ids = list(successful_optimizations_frame['optimization_id'])
all_trades_frame = analyzer.compare_optimizations(successful_ids, 'trades')

In [33]:
# You might want to incorporate constraint thresholds to the trades frames to pivot on constraint types

In [34]:
portfolio = 'EQUITY8_US'

In [35]:
trade_stats_frame = analyzer.get_security_trade_stats(all_trades_frame, successful_optimizations_frame, portfolio)

In [36]:
trade_stats_frame

trade_type,Buys,Liquidated Holdings,New Holdings,Sells,buy_consistency,sell_consistency,new_holdings_consistency,total_actions
ticker,,,,,,,,
A US,88,340,0,0,0.146179,0.564784,0.0,428
AA US,0,602,0,0,0.000000,1.000000,0.0,602
AAL US,2,600,0,0,0.003322,0.996678,0.0,602
AAON US,86,516,0,0,0.142857,0.857143,0.0,602
AAP US,0,602,0,0,0.000000,1.000000,0.0,602
...,...,...,...,...,...,...,...,...
ZION US,10,592,0,0,0.016611,0.983389,0.0,602
ZM US,94,508,0,0,0.156146,0.843854,0.0,602
ZS US,62,540,0,0,0.102990,0.897010,0.0,602


In [37]:
trade_stats_frame.sort_values('Buys',ascending=False)['Buys'].unique()

array([602, 598, 596, 592, 583, 580, 579, 577, 576, 575, 574, 570, 564,
       563, 562, 558, 556, 555, 545, 539, 532, 529, 524, 521, 504, 501,
       491, 486, 482, 476, 470, 460, 450, 449, 443, 441, 440, 434, 433,
       427, 426, 419, 415, 411, 410, 407, 398, 390, 384, 380, 375, 371,
       369, 363, 360, 358, 357, 353, 350, 342, 330, 328, 327, 314, 310,
       307, 304, 303, 301, 300, 298, 297, 295, 289, 285, 281, 278, 274,
       268, 266, 260, 257, 254, 253, 250, 241, 234, 233, 232, 229, 228,
       225, 224, 218, 215, 214, 213, 212, 206, 204, 199, 198, 196, 194,
       191, 187, 186, 184, 181, 179, 177, 175, 174, 172, 170, 166, 165,
       162, 159, 155, 154, 152, 150, 148, 147, 145, 144, 141, 140, 138,
       136, 135, 131, 130, 129, 127, 126, 125, 124, 123, 122, 121, 120,
       118, 116, 115, 114, 112, 111, 109, 108, 107, 106, 105, 104, 103,
       102, 101, 100,  99,  98,  97,  96,  94,  92,  91,  90,  89,  88,
        87,  86,  83,  82,  81,  80,  79,  77,  76,  75,  74,  7

In [38]:
# Calculate consistency metrics from your trade_stats_frame

# X-Axis: Buy Consistency Ratio (0 to 1)

# 0: The security was never bought in any optimization
# 1: The security was bought in every optimization
# 0.5: The security was bought in 50% of the optimizations

# Y-Axis: Sell Consistency Ratio (0 to 1)

# 0: The security was never sold in any optimization
# 1: The security was sold in every optimization
# 0.5: The security was sold in 50% of the optimizations

# What the positions tell us:

# Bottom Left (0,0): Securities that were rarely or never traded - these are "passive" holdings
# Top Right (1,1): Securities that were both frequently bought AND sold - this would be unusual and might indicate volatility in the optimizer's decisions
# Bottom Right (1,0): Securities consistently bought but rarely sold - these are "favorites"
# Top Left (0,1): Securities consistently sold but rarely bought - these are "avoided" securities

# Additional dimensions:

# Size of dots: Represents total trading activity (sum of all buy/sell/new/liquidated actions)
# Color: Could represent new holdings consistency or other metrics

# This visualization helps identify:

# Which securities the optimizer consistently favors (buys)
# Which securities the optimizer consistently avoids (sells)
# Which securities show mixed or inconsistent behavior
# The overall trading patterns in your optimization strategy

In [45]:
trade_analytics_html = ipw.HTML("<h2>Optimization Trade Analytics</2>")

security_action_frequency_accordion = ipw.Accordion(children=[plot_manager.build_security_action_frequency_fig(trade_stats_frame, portfolio)])
security_action_frequency_accordion.set_title(0, "Security Action Frequency")
security_action_frequency_accordion.selected_index = None

security_trade_consistency_accordion = ipw.Accordion(children=[build_security_trade_consistency_fig(trade_stats_frame, portfolio)])
security_trade_consistency_accordion.set_title(0, "Security Trade Consistency")
security_trade_consistency_accordion.selected_index = None

security_action_distribution_accordion = ipw.Accordion(children=[plot_manager.build_security_actions_treemap(trade_stats_frame, portfolio)])
security_action_distribution_accordion.set_title(0, "Security Action Distribution")
security_action_distribution_accordion.selected_index = None

In [46]:
ipw.VBox([
    trade_analytics_html,
    security_action_frequency_accordion,
    security_trade_consistency_accordion,
    security_action_distribution_accordion
])

    'dat…

<h2>7. Advanced Analytics</h2>

Monte Carlo Simulation: Results distribution from varying input parameters<br>
What-If Scenario Builder: Tool to test custom constraint configurations<br>
Correlation Dashboard: Shows relationships between different parameters and outcomes<br>
Optimization Parameter Importance: ML-based analysis of which parameters have the most impact

In [41]:
# Build Correlation Dashboard similar to how you build Constraint Sensitivity Analysis:
    # 1 View with no filters
    # 1 View with a single constraint filter
    # 1 View with 2 constraint filters

<h2>8. User Interaction Features</h2>

Parameter Selection Controls: Interactive filters for constraints, goals, etc.<br>
Optimization Comparison Tool: Side-by-side comparison of selected optimizations<br>
Bookmark & Export Functionality: Save and export interesting configurations

In [42]:
metadata_constraint_mapper = {
        ('weight', 'GICS Sector:All'): 'max_sector_deviation'
    }

# Apply custom CSS to make the readout boxes wider
custom_css = """
<style>
.widget-readout {
    min-width: 100px !important;
    width: auto !important;
}
</style>
"""

# Display the custom CSS and the slider
display(HTML(custom_css))

In [43]:
frame = workflow_mngr.tracker.get_successful_optimizations()
frame = workflow_mngr.tracker.filter_optimizations(
    max_security_weight=0.05, 
    max_positions=200, 
    max_turnover=0.3, 
    portfolio=portfolio, 
    status='success'
)

optimization_ids = list(frame['optimization_id'])


results_frame = analyzer.compare_optimizations(optimization_ids, comparison_type="optimization_pathways").rename(columns=custom_goal_mapper)

data_manager = OptimizationDataManager(results_frame)
gui = PortfolioOptimizationGUI(data_manager)

In [44]:
gui.view

1. Bookmark Optimizations
2. Generate a new config file for that task over different dates
3. Cataloge tasks in a separate index
4. Register optimizations with committed trades
5. Run Attribution on the optimal portfolio (need to use a temp portfolio)
6. Compare to actual portfolio attribution

3. Portfolio Characteristic Analysis

Sector Allocation Dashboard: Shows sector weights across different optimizations<br>
Factor Exposure Comparison: Compare factor tilts (value, momentum, quality, etc.) across optimizations<br>
Stock Contribution Analysis: Identify consistently selected/excluded securities

4. Risk Model Sensitivity

Risk Model Comparison: Toggle between different risk models to see impact<br>
Time Horizon Impact: How different time horizons affect the optimization results<br>
Risk Decomposition: Break down total risk into factor and specific risk components<br>
Scaling Effects: Impact of different risk scaling methods

5. Optimization Goal Analysis

Goal Trade-off Visualizer: Interactive tool showing the effect of adjusting goal weights<br>
Pareto Frontier: Plot showing the trade-off between competing objectives<br>
Goal Achievement Metrics: How well each optimization achieves its stated goals<br>
Sensitivity Sliders: Interactive controls to adjust goal parameters and see results change

6. Time Series Analysis

Optimization Stability: Compare optimization results across different as-of dates<br>
Performance Backtest: Simulated performance of optimized portfolios<br>
Turnover Trend: Analysis of turnover requirements over time<br>
Risk Forecast Evolution: How predicted risk evolves over time